In [8]:
import pandas as pd
import numpy as np
import re
import unicodedata

In [68]:
df = pd.read_csv('datos\\Dataset_Gusano_Barrenador_Completo.csv')
df

,Event_ID,Outbreak_ID,Latitud,Longitud,Fecha_Inicio,Municipio,Descripcion,Especies_Involucradas,Total_Casos_Evento,Total_Muertes_Evento
0,7430,188989,22.446768,-101.548092,2026-04-27,Pinos,Bovino de 7 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
1,7430,188990,24.422818,-100.927736,2026-05-01,El Salvador,Bovino de 8 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
2,7430,188991,24.462645,-101.058128,2026-05-01,Concepción del Oro,Ovino de 5 años de edad con miasis en región d...,"Bovinos (6), Perros (4), Ovinos (1)",11,0
3,7430,188992,24.432582,-101.060543,2026-05-01,Concepción del Oro,Canino de 10 años de edad con miasis en región...,"Bovinos (6), Perros (4), Ovinos (1)",11,0
4,7430,188199,22.153161,-101.712401,2026-04-23,Pinos,Canino de 5 años con miasis en región cervical.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
...,...,...,...,...,...,...,...,...,...,...
3299,6269,150994,17.784750,-91.036623,2025-02-15,Balancán,"Bovino hembra de 7 años de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3300,6269,150995,17.784750,-91.036623,2025-02-15,Balancán,"Bovino hembra de 7 años de edad, con miasis en...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3301,6269,150990,17.849839,-91.137891,2025-02-14,Balancán,"Bovino hembra de 3 días de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3302,6269,150991,17.917676,-91.172024,2025-02-14,Balancán,"Bovino hembra de 5 días de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0


In [ ]:
def clean_key(text):
    nfkd = unicodedata.normalize('NFKD', str(text))
    sin_tildes = "".join([c for c in nfkd if not unicodedata.combining(c)])
    limpio = re.sub(r'[^a-zA-Z0-9]+', '_', sin_tildes.lower()).strip('_')
    return f"casos_{limpio}"

def clean_municipio(text):
    if pd.isna(text):
        return "Sin especificar"
    limpio = re.sub(r'\s*\([\d\-\s]+\)$', '', str(text))
    return limpio.strip()

item_pattern = re.compile(r'^(.*?)\s*\((\d+)\)$')

def parse_animal_counts(cell_text):
    if not isinstance(cell_text, str) or not cell_text.strip():
        return {}
    
    counts = {}
    for item in cell_text.split(','):
        match = item_pattern.match(item.strip())
        if match:
            nombre, cantidad = match.groups()
            counts[clean_key(nombre)] = int(cantidad)
    return counts

df['Fecha_Inicio'] = pd.to_datetime(df['Fecha_Inicio'], dayfirst=True, errors='coerce')
df['Municipio_Limpio'] = df['Municipio'].apply(clean_municipio)
df['Coordenadas'] = df['Latitud'].astype(str) + ", " + df['Longitud'].astype(str)

df_eventos = df.groupby('Event_ID', as_index=False).agg(
    Fecha_Inicio_Evento=('Fecha_Inicio', 'min'),
    Fecha_Ultimo_Brote=('Fecha_Inicio', 'max'),
    Total_Brotes_Asociados=('Outbreak_ID', 'count'),
    Municipios_Involucrados=('Municipio_Limpio', lambda x: ', '.join(sorted(x.unique()))),
    Especies_Involucradas=('Especies_Involucradas', 'first'),
    Total_Casos_Evento=('Total_Casos_Evento', 'first'),
    Total_Muertes_Evento=('Total_Muertes_Evento', 'first')
)

df_especies_cols = (
    df_eventos['Especies_Involucradas']
    .apply(parse_animal_counts)
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)

df_eventos_final = pd.concat([df_eventos, df_especies_cols], axis=1)
df_eventos_final = df_eventos_final.sort_values('Fecha_Inicio_Evento')

df_eventos_final.to_csv('2_tabla_eventos_casos_y_especies.csv', index=False, encoding='utf-8-sig')

In [ ]:






# --- FUNCIONES DE LIMPIEZA Y EXTRACCIÓN ---

def clean_key(text):
    """Normaliza nombres de especies para usarlos como nombres de columnas."""
    nfkd = unicodedata.normalize('NFKD', str(text))
    sin_tildes = "".join([c for c in nfkd if not unicodedata.combining(c)])
    # Reemplazar espacios y caracteres no alfanuméricos por guion bajo
    limpio = re.sub(r'[^a-zA-Z0-9]+', '_', sin_tildes.lower()).strip('_')
    return f"casos_{limpio}"

def clean_municipio(text):
    """Limpia períodos entre paréntesis como 'Colon (2005-2009)' -> 'Colon'."""
    if pd.isna(text):
        return "Sin especificar"
    # Elimina paréntesis que contengan rangos de años o números al final
    limpio = re.sub(r'\s*\([\d\-\s]+\)$', '', str(text))
    return limpio.strip()

# Patrón para capturar 'Especie (Cantidad)' al final de cada bloque
item_pattern = re.compile(r'^(.*?)\s*\((\d+)\)$')

def parse_animal_counts(cell_text):
    """Convierte el texto de especies en un diccionario {especie: cantidad}."""
    if not isinstance(cell_text, str) or not cell_text.strip():
        return {}
    
    counts = {}
    for item in cell_text.split(','):
        match = item_pattern.match(item.strip())
        if match:
            nombre, cantidad = match.groups()
            counts[clean_key(nombre)] = int(cantidad)
    return counts

# --- PREPROCESAMIENTO GENERAL ---

# Convertir Fecha_Inicio a formato estándar YYYY-MM-DD (importante para Looker Studio)
df['Fecha_Inicio'] = pd.to_datetime(df['Fecha_Inicio'], dayfirst=True, errors='coerce')

# Limpiar nombres de municipios
df['Municipio_Limpio'] = df['Municipio'].apply(clean_municipio)

# Crear coordenada combinada "Latitud, Longitud" (formato nativo de mapas en Looker Studio)
df['Coordenadas'] = df['Latitud'].astype(str) + ", " + df['Longitud'].astype(str)


# ==============================================================================
# TABLA 1: NIVEL BROTE / FOCO (Para Mapas, Serie Temporal de Focos y Filtro Municipio)
# ==============================================================================

cols_brotes = [
    'Outbreak_ID', 
    'Event_ID', 
    'Fecha_Inicio', 
    'Municipio', 
    'Municipio_Limpio', 
    'Latitud', 
    'Longitud', 
    'Coordenadas', 
    'Descripcion'
]

df_brotes = df[cols_brotes].copy()
df_brotes = df_brotes.sort_values(['Fecha_Inicio', 'Outbreak_ID'])


# ==============================================================================
# TABLA 2: NIVEL EVENTO ÚNICO (Para KPIs de Casos, Muertes y Desglose por Especie)
# ==============================================================================

# Consolidar cada Event_ID sin duplicar las sumas
df_eventos = df.groupby('Event_ID', as_index=False).agg(
    Fecha_Inicio_Evento=('Fecha_Inicio', 'min'),
    Fecha_Ultimo_Brote=('Fecha_Inicio', 'max'),
    Total_Brotes_Asociados=('Outbreak_ID', 'count'),
    Municipios_Involucrados=('Municipio_Limpio', lambda x: ', '.join(sorted(x.unique()))),
    Especies_Involucradas=('Especies_Involucradas', 'first'),
    Total_Casos_Evento=('Total_Casos_Evento', 'first'),
    Total_Muertes_Evento=('Total_Muertes_Evento', 'first')
)

# Extraer las columnas de cada especie a partir de 'Especies_Involucradas'
df_especies_cols = (
    df_eventos['Especies_Involucradas']
    .apply(parse_animal_counts)
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)

# Unir el resumen del evento con las nuevas columnas de especies
df_eventos_final = pd.concat([df_eventos, df_especies_cols], axis=1)
df_eventos_final = df_eventos_final.sort_values('Fecha_Inicio_Evento')


# --- EXPORTACIÓN ---
# Exportamos con utf-8-sig para que Excel y Looker Studio reconozcan acentos y 'ñ' sin problema
df_brotes.to_csv('1_tabla_brotes_mapa_y_tiempo.csv', index=False, encoding='utf-8-sig')
df_eventos_final.to_csv('2_tabla_eventos_casos_y_especies.csv', index=False, encoding='utf-8-sig')

print(f"Tabla Brotes generada: {len(df_brotes)} filas.")
print(f"Tabla Eventos generada: {len(df_eventos_final)} eventos únicos.")

C:\Users\109149\AppData\Local\Temp\ipykernel_25228\3684299535.py:38: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['Fecha_Inicio'] = pd.to_datetime(df['Fecha_Inicio'], dayfirst=True, errors='coerce')


Tabla Brotes generada: 3304 filas.
Tabla Eventos generada: 47 eventos únicos.


In [67]:
GM = pd.read_csv('datos\\Dataset_Gusano_Barrenador_Completo.csv')
GM

,Event_ID,Outbreak_ID,Latitud,Longitud,Fecha_Inicio,Municipio,Descripcion,Especies_Involucradas,Total_Casos_Evento,Total_Muertes_Evento
0,7430,188989,22.446768,-101.548092,2026-04-27,Pinos,Bovino de 7 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
1,7430,188990,24.422818,-100.927736,2026-05-01,El Salvador,Bovino de 8 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
2,7430,188991,24.462645,-101.058128,2026-05-01,Concepción del Oro,Ovino de 5 años de edad con miasis en región d...,"Bovinos (6), Perros (4), Ovinos (1)",11,0
3,7430,188992,24.432582,-101.060543,2026-05-01,Concepción del Oro,Canino de 10 años de edad con miasis en región...,"Bovinos (6), Perros (4), Ovinos (1)",11,0
4,7430,188199,22.153161,-101.712401,2026-04-23,Pinos,Canino de 5 años con miasis en región cervical.,"Bovinos (6), Perros (4), Ovinos (1)",11,0
...,...,...,...,...,...,...,...,...,...,...
3299,6269,150994,17.784750,-91.036623,2025-02-15,Balancán,"Bovino hembra de 7 años de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3300,6269,150995,17.784750,-91.036623,2025-02-15,Balancán,"Bovino hembra de 7 años de edad, con miasis en...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3301,6269,150990,17.849839,-91.137891,2025-02-14,Balancán,"Bovino hembra de 3 días de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0
3302,6269,150991,17.917676,-91.172024,2025-02-14,Balancán,"Bovino hembra de 5 días de edad, con presencia...","Gavilán común (1), Aves (1), Bovinos (443), Pe...",560,0


In [7]:
GM['Especies_Involucradas'].str.lower().str.split(',')

0                 [bovinos (6),  perros (4),  ovinos (1)]
1                 [bovinos (6),  perros (4),  ovinos (1)]
2                 [bovinos (6),  perros (4),  ovinos (1)]
3                 [bovinos (6),  perros (4),  ovinos (1)]
4                 [bovinos (6),  perros (4),  ovinos (1)]
                              ...                        
3299    [gavilán común (1),  aves (1),  bovinos (443),...
3300    [gavilán común (1),  aves (1),  bovinos (443),...
3301    [gavilán común (1),  aves (1),  bovinos (443),...
3302    [gavilán común (1),  aves (1),  bovinos (443),...
3303    [gavilán común (1),  aves (1),  bovinos (443),...
Name: Especies_Involucradas, Length: 3304, dtype: object

In [ ]:
def clean_key(text):
    nfkd = unicodedata.normalize('NFKD', text)
    sin_tildes = "".join([c for c in nfkd if not unicodedata.combining(c)])
    return sin_tildes.lower().strip()

# Patrón regex:
# - r'^(.*?)\s*\((\d+)\)$'
# Toma todo el texto hasta el último conjunto de paréntesis con dígitos (\d+).
item_pattern = re.compile(r'^(.*?)\s*\((\d+)\)$')

def parse_animal_counts(cell_text):
    if not isinstance(cell_text, str) or not cell_text.strip():
        return {}
    
    counts = {}
    items = cell_text.split(',')
    
    for item in items:
        item = item.strip()
        match = item_pattern.match(item)
        if match:
            nombre, cantidad = match.groups()
            key = clean_key(nombre)
            counts[key] = int(cantidad)
            
    return counts

In [12]:
GM_animales = GM['Especies_Involucradas'].apply(parse_animal_counts).apply(pd.Series).fillna(0).astype(int)
GM_animales

,bovinos,perros,ovinos,equidos,suidos,caprinos,conejos,aves,gatos,ovinos y caprinos (rebano mixto),mono caraya negro,bufalos,tlacuache sureno,leon,puma,venado cola blanca,mapache,gavilan comun
0,6,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,6,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,6,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,6,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,6,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3299,443,27,5,54,27,2,0,1,0,0,0,0,0,0,0,0,0,1
3300,443,27,5,54,27,2,0,1,0,0,0,0,0,0,0,0,0,1
3301,443,27,5,54,27,2,0,1,0,0,0,0,0,0,0,0,0,1
3302,443,27,5,54,27,2,0,1,0,0,0,0,0,0,0,0,0,1


In [14]:
GM_unido = pd.concat([GM, GM_animales], axis=1)
GM_unido.head()

,Event_ID,Outbreak_ID,Latitud,Longitud,Fecha_Inicio,Municipio,Descripcion,Especies_Involucradas,Total_Casos_Evento,Total_Muertes_Evento,...,gatos,ovinos y caprinos (rebano mixto),mono caraya negro,bufalos,tlacuache sureno,leon,puma,venado cola blanca,mapache,gavilan comun
0,7430,188989,22.446768,-101.548092,2026-04-27,Pinos,Bovino de 7 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0
1,7430,188990,24.422818,-100.927736,2026-05-01,El Salvador,Bovino de 8 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0
2,7430,188991,24.462645,-101.058128,2026-05-01,Concepción del Oro,Ovino de 5 años de edad con miasis en región d...,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0
3,7430,188992,24.432582,-101.060543,2026-05-01,Concepción del Oro,Canino de 10 años de edad con miasis en región...,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0
4,7430,188199,22.153161,-101.712401,2026-04-23,Pinos,Canino de 5 años con miasis en región cervical.,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
GM_unido.to_csv("GB_animales.csv", index=False, encoding='utf-8-sig')

In [46]:
len(GM['Event_ID'].unique())

47

In [22]:
GM_eventos_unicos = GM_unido.drop_duplicates(subset=['Event_ID'])
GM_eventos_unicos.head()

,Event_ID,Outbreak_ID,Latitud,Longitud,Fecha_Inicio,Municipio,Descripcion,Especies_Involucradas,Total_Casos_Evento,Total_Muertes_Evento,...,gatos,ovinos y caprinos (rebano mixto),mono caraya negro,bufalos,tlacuache sureno,leon,puma,venado cola blanca,mapache,gavilan comun
0,7430,188989,22.446768,-101.548092,2026-04-27,Pinos,Bovino de 7 días de edad con miasis en ombligo.,"Bovinos (6), Perros (4), Ovinos (1)",11,0,...,0,0,0,0,0,0,0,0,0,0
11,7431,188886,21.142810,-104.954130,2026-04-28,Compostela,Equino de 2 años de edad con miasis en miembro...,"Bovinos (6), Perros (1), Équidos (2)",9,0,...,0,0,0,0,0,0,0,0,0,0
20,5127,131967,10.782200,-84.375800,2024-02-03,San Carlos,NaN,"Bovinos (48), Perros (11), Équidos (1), Ovinos...",64,1,...,0,0,0,0,0,0,0,0,0,0
72,6289,161179,18.271080,-88.638710,2025-07-08,Othón P. Blanco,Bovino de 4 años de edad con miasis en región ...,"Bovinos (65), Perros (2), Équidos (2), Suidos (4)",73,0,...,0,0,0,0,0,0,0,0,0,0
141,6163,161077,16.136490,-88.844870,2025-07-04,Toledo,8-year-old pig with NWS myiasis in the rear ri...,"Bovinos (43), Perros (85), Équidos (17), Capri...",180,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
GM_eventos_unicos['Total_Casos_Evento'].sum()
# len(GM_eventos_unicos)

np.int64(12707)

-------

In [59]:
# Consolidar cada evento con su fecha de inicio más temprana
eventos_consolidados = GM.groupby('Event_ID').agg({
    'Fecha_Inicio': 'min',
    'Total_Casos_Evento': 'first'  # o 'max', al ser idéntico en todas las filas del evento
}).reset_index()

# Incidencia diaria por fecha de inicio del evento
casos_por_dia = (
    eventos_consolidados.groupby('Fecha_Inicio')['Total_Casos_Evento']
    .sum()
    .reset_index(name='Casos_Por_Dia')
    .sort_values('Fecha_Inicio')
)

casos_por_dia.head()

,Fecha_Inicio,Casos_Por_Dia
0,2009-04-18,9
1,2016-07-13,144
2,2019-12-18,1
3,2021-06-29,6
4,2022-07-19,1


In [56]:
casos_por_dia['Casos_Por_Eventos_Iniciados'].sum()

np.int64(12707)

In [60]:
eventos_consolidados = GM.groupby('Event_ID').agg({
    'Fecha_Inicio': 'min',
    'Total_Muertes_Evento': 'first'  # o 'max', al ser idéntico en todas las filas del evento
}).reset_index()

# Incidencia diaria por fecha de inicio del evento
muertes_por_dia = (
    eventos_consolidados.groupby('Fecha_Inicio')['Total_Muertes_Evento']
    .sum()
    .reset_index(name='Muertes_Por_Dia')
    .sort_values('Fecha_Inicio')
)

muertes_por_dia.head()

,Fecha_Inicio,Muertes_Por_Dia
0,2009-04-18,0
1,2016-07-13,137
2,2019-12-18,0
3,2021-06-29,0
4,2022-07-19,0


In [66]:
reporte_diario = pd.merge(
    casos_por_dia,
    muertes_por_dia,
    on='Fecha_Inicio',
    how='outer'
).sort_values('Fecha_Inicio')

# Rellenar con 0 los días donde no hubo registros en alguna métrica
reporte_diario[['Casos_Por_Dia', 'Muertes_Por_Dia']] = (
    reporte_diario[['Casos_Por_Dia', 'Muertes_Por_Dia']].fillna(0).astype(int)
)

reporte_diario

,Fecha_Inicio,Casos_Por_Dia,Muertes_Por_Dia
0,2009-04-18,9,0
1,2016-07-13,144,137
2,2019-12-18,1,0
3,2021-06-29,6,0
4,2022-07-19,1,0
5,2022-08-16,2968,0
6,2023-06-01,4650,0
7,2023-07-14,64,1
8,2024-09-10,765,1
9,2024-10-25,1017,2


In [65]:
reporte_diario.to_csv("reporte_evento_diario.csv", index=False, encoding='utf-8-sig')

--------

In [33]:
GB = pd.read_csv('datos\\GBG_diario_pais_provincia_2022_2026.csv')
GB

,fecha,pais,provincia,numero_focos,casos_reportados,focos_sin_total_casos,casos_caninos,casos_equinos,casos_porcinos,casos_bovinos,casos_ovinos,casos_aves_corral,casos_caprinos,casos_felinos,casos_bufalos,casos_aves_silvestres,casos_fauna_silvestre,casos_conejos,fuente,url_fuente
0,2022-08-16,PANAMÁ,Panamá,1,2.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
1,2022-09-10,PANAMÁ,Panamá,1,1.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
2,2022-10-21,PANAMÁ,Panamá,1,1.0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
3,2022-12-04,PANAMÁ,Panamá,1,1.0,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
4,2022-12-15,PANAMÁ,Panamá,1,461.0,0,21.0,4.0,17.0,418.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,2026-07-28,EUA,Texas,1,2.0,0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
730,2026-07-31,EUA,Texas,1,1.0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
731,2026-08-04,EUA,Texas,1,1.0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...
732,2026-08-14,EUA,Texas,1,1.0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SENASICA GIS / OMSA,https://dj.senasica.gob.mx/mapas/rest/services...


In [42]:
GB['fecha'] = pd.to_datetime(GB['fecha'], errors = 'coerce')

In [ ]:
casos_dia = GB.groupby('fecha')['casos_reportados'].sum().reset_index(name='casos_dia')
casos_dia

,fecha,Casos_dia
0,2022-08-16,2.0
1,2022-09-10,1.0
2,2022-10-21,1.0
3,2022-12-04,1.0
4,2022-12-15,461.0
...,...,...
331,2026-07-28,2.0
332,2026-07-31,1.0
333,2026-08-04,1.0
334,2026-08-14,1.0
